# 1. Analyse exploratoire

Chargement des données, périmètre (bâtiments non résidentiels), nettoyage et choix des colonnes.

**Sortie :** `data/batiments_nettoyes.csv`, lu par le notebook 2.

### Import des modules

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

### Analyse Exploratoire

In [2]:
building_consumption = pd.read_csv("data/2016_Building_Energy_Benchmarking.csv")

In [3]:
# On regarde comment un batiment est défini dans ce jeu de données 
building_consumption.head()

,OSEBuildingID,DataYear,BuildingType,PrimaryPropertyType,PropertyName,Address,City,State,ZipCode,TaxParcelIdentificationNumber,...,Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),DefaultData,Comments,ComplianceStatus,Outlier,TotalGHGEmissions,GHGEmissionsIntensity
0,1,2016,NonResidential,Hotel,Mayflower park hotel,405 Olive way,Seattle,WA,98101.0,0659000030,...,1.156514e+06,3946027.0,12764.52930,1276453.0,False,NaN,Compliant,NaN,249.98,2.83
1,2,2016,NonResidential,Hotel,Paramount Hotel,724 Pine street,Seattle,WA,98101.0,0659000220,...,9.504252e+05,3242851.0,51450.81641,5145082.0,False,NaN,Compliant,NaN,295.86,2.86
2,3,2016,NonResidential,Hotel,5673-The Westin Seattle,1900 5th Avenue,Seattle,WA,98101.0,0659000475,...,1.451544e+07,49526664.0,14938.00000,1493800.0,False,NaN,Compliant,NaN,2089.28,2.19
3,5,2016,NonResidential,Hotel,HOTEL MAX,620 STEWART ST,Seattle,WA,98101.0,0659000640,...,8.115253e+05,2768924.0,18112.13086,1811213.0,False,NaN,Compliant,NaN,286.43,4.67
4,8,2016,NonResidential,Hotel,WARWICK SEATTLE HOTEL (ID8),401 LENORA ST,Seattle,WA,98121.0,0659000970,...,1.573449e+06,5368607.0,88039.98438,8803998.0,False,NaN,Compliant,NaN,505.01,2.88


In [4]:
# On regarde le nombre de valeurs manquantes par colonne ainsi que leur type 
building_consumption.info()

<class 'pandas.DataFrame'>
RangeIndex: 3376 entries, 0 to 3375
Data columns (total 46 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   OSEBuildingID                    3376 non-null   int64  
 1   DataYear                         3376 non-null   int64  
 2   BuildingType                     3376 non-null   str    
 3   PrimaryPropertyType              3376 non-null   str    
 4   PropertyName                     3376 non-null   str    
 5   Address                          3376 non-null   str    
 6   City                             3376 non-null   str    
 7   State                            3376 non-null   str    
 8   ZipCode                          3360 non-null   float64
 9   TaxParcelIdentificationNumber    3376 non-null   str    
 10  CouncilDistrictCode              3376 non-null   int64  
 11  Neighborhood                     3376 non-null   str    
 12  Latitude                       

In [5]:
building_consumption["BuildingType"].value_counts()

BuildingType
NonResidential          1460
Multifamily LR (1-4)    1018
Multifamily MR (5-9)     580
Multifamily HR (10+)     110
SPS-District K-12         98
Nonresidential COS        85
Campus                    24
Nonresidential WA          1
Name: count, dtype: int64

In [6]:
building_consumption["PrimaryPropertyType"].value_counts()

PrimaryPropertyType
Low-Rise Multifamily           987
Mid-Rise Multifamily           564
Small- and Mid-Sized Office    293
Other                          256
Warehouse                      187
Large Office                   173
K-12 School                    139
Mixed Use Property             133
High-Rise Multifamily          105
Retail Store                    91
Hotel                           77
Worship Facility                71
Distribution Center             53
Senior Care Community           45
Supermarket / Grocery Store     40
Medical Office                  39
Self-Storage Facility           28
University                      25
Residence Hall                  23
Refrigerated Warehouse          12
Restaurant                      12
Hospital                        10
Laboratory                      10
Office                           3
Name: count, dtype: int64

In [7]:
n_avant = len(building_consumption)
df = building_consumption[~building_consumption["BuildingType"].str.contains("Multifamily")]

In [8]:
print(f"{n_avant} -> {len(df)} bâtiments ({n_avant - len(df)} retirés)")

3376 -> 1668 bâtiments (1708 retirés)


In [9]:
# % de cases vides par colonne, du plus vide au moins vide
manquants = df.isna().mean().sort_values(ascending=False) * 100
manquants[manquants > 0].round(1)

Comments                           100.0
Outlier                             99.0
YearsENERGYSTARCertified            94.1
ThirdLargestPropertyUseType         78.8
ThirdLargestPropertyUseTypeGFA      78.8
SecondLargestPropertyUseType        48.7
SecondLargestPropertyUseTypeGFA     48.7
ENERGYSTARScore                     34.4
ZipCode                              1.0
LargestPropertyUseType               0.4
LargestPropertyUseTypeGFA            0.4
SiteEUI(kBtu/sf)                     0.2
SiteEUIWN(kBtu/sf)                   0.2
SiteEnergyUseWN(kBtu)                0.2
Electricity(kBtu)                    0.1
Electricity(kWh)                     0.1
NaturalGas(kBtu)                     0.1
NaturalGas(therms)                   0.1
GHGEmissionsIntensity                0.1
SteamUse(kBtu)                       0.1
SourceEUIWN(kBtu/sf)                 0.1
SourceEUI(kBtu/sf)                   0.1
TotalGHGEmissions                    0.1
NumberofBuildings                    0.1
ListOfAllPropert

In [10]:
n_avant = len(df)

# 1. On ne garde que les relevés jugés fiables par la ville
df = df[df["ComplianceStatus"] == "Compliant"]

# 2. Un bâtiment sans 2e/3e usage = 0 m² pour cet usage
for col in ["SecondLargestPropertyUseTypeGFA", "ThirdLargestPropertyUseTypeGFA"]:
    df[col] = df[col].fillna(0)

print(f"{n_avant} -> {len(df)} bâtiments")

1668 -> 1548 bâtiments


In [11]:
# vérif : il ne reste plus de bâtiments signalés Outlier ou avec des données par défaut
print(df["Outlier"].value_counts(dropna=False))
print(df["DefaultData"].value_counts())

Outlier
NaN    1548
Name: count, dtype: int64
DefaultData
False    1548
Name: count, dtype: int64


In [12]:
# Colonnes retirées, rangées par raison
a_supprimer = [
    # 1. Identifiants et adresse : ils désignent un bâtiment sans rien dire de sa consommation
    "OSEBuildingID", "DataYear", "PropertyName", "Address", "City", "State",
    "ZipCode", "TaxParcelIdentificationNumber", "CouncilDistrictCode",
    # 2. Vides ou sans information utile
    "Comments", "YearsENERGYSTARCertified", "Outlier", "ListOfAllPropertyUseTypes",
    "ComplianceStatus", "DefaultData",
    # 3. Déjà utilisées ou peu utiles (retour mentor) : BuildingType a servi au filtre du début
    "BuildingType", "Latitude", "Longitude",
    # 4. Fuite de données : ces valeurs ne sont connues qu'après avoir mesuré la consommation
    "SiteEUI(kBtu/sf)", "SiteEUIWN(kBtu/sf)", "SourceEUI(kBtu/sf)",
    "SourceEUIWN(kBtu/sf)", "SiteEnergyUseWN(kBtu)", "GHGEmissionsIntensity",
]
# Gardées pour le feature engineering : ENERGYSTARScore, SteamUse, Electricity, NaturalGas

df = df.drop(columns=a_supprimer)
print(df.shape)
df.columns

(1548, 22)


Index(['PrimaryPropertyType', 'Neighborhood', 'YearBuilt', 'NumberofBuildings',
       'NumberofFloors', 'PropertyGFATotal', 'PropertyGFAParking',
       'PropertyGFABuilding(s)', 'LargestPropertyUseType',
       'LargestPropertyUseTypeGFA', 'SecondLargestPropertyUseType',
       'SecondLargestPropertyUseTypeGFA', 'ThirdLargestPropertyUseType',
       'ThirdLargestPropertyUseTypeGFA', 'ENERGYSTARScore',
       'SiteEnergyUse(kBtu)', 'SteamUse(kBtu)', 'Electricity(kWh)',
       'Electricity(kBtu)', 'NaturalGas(therms)', 'NaturalGas(kBtu)',
       'TotalGHGEmissions'],
      dtype='str')

#### TERMINER L'ANALYSE EXPLORATOIRE 

A réaliser : 
- Une analyse descriptive des données, y compris une explication du sens des colonnes gardées, des arguments derrière la suppression de lignes ou de colonnes, des statistiques descriptives et des visualisations pertinentes.

Qelques pistes d'analyse : 

* Identifier les colonnes avec une majorité de valeurs manquantes ou constantes en utilisant la méthode value_counts() de Pandas
* Mettre en evidence les différences entre les immeubles mono et multi-usages
* Utiliser des pairplots et des boxplots pour faire ressortir les outliers ou des batiments avec des valeurs peu cohérentes d'un point de vue métier 

Pour vous inspirer, ou comprendre l'esprit recherché dans une analyse exploratoire, vous pouvez consulter ce notebook en ligne : https://www.kaggle.com/code/pmarcelino/comprehensive-data-exploration-with-python. Il ne s'agit pas d'un modèle à suivre à la lettre ni d'un template d'analyses attendues pour ce projet. 

### Sauvegarde pour le notebook suivant

In [13]:
df.to_csv("data/batiments_nettoyes.csv", index=False)
print(df.shape)

(1548, 22)
